# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [1]:
# Goal: summarize two FlyRank paper findings, label source, and validation questions.
findings = [
    {
        "finding": "The label comes from whether search impressions declined, i.e. trend_direction == 'down'.",
        "label_source": "The label is derived from `trend_direction`, which is calculated from last-30d vs prev-30d impressions.",
        "validation_question": (
            "This claim is carried only if the model never uses `trend_direction` or `trend_pct`, "
            "and if the split keeps future/held-out clients separate from training."
        ),
    },
    {
        "finding": "A refresh opportunity model should use observable page and traffic signals to rank pages at risk of decline.",
        "label_source": "The label is the observed downstream decline state, not a product action label or refresh flag.",
        "validation_question": (
            "The validation design should be client-aware or time-aware, so performance reflects new pages and clients rather than memorized page history."
        ),
    },
]
findings

[{'finding': "The label comes from whether search impressions declined, i.e. trend_direction == 'down'.",
  'label_source': 'The label is derived from `trend_direction`, which is calculated from last-30d vs prev-30d impressions.',
  'validation_question': 'This claim is carried only if the model never uses `trend_direction` or `trend_pct`, and if the split keeps future/held-out clients separate from training.'},
 {'finding': 'A refresh opportunity model should use observable page and traffic signals to rank pages at risk of decline.',
  'label_source': 'The label is the observed downstream decline state, not a product action label or refresh flag.',
  'validation_question': 'The validation design should be client-aware or time-aware, so performance reflects new pages and clients rather than memorized page history.'}]

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [3]:
# Goal: rerun the Week-5 model under an honest client-aware split and compare to the original split.
import sys
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_score, recall_score, roc_auc_score

repo_root = Path.cwd()
while repo_root.name != "flyrank-internship" and repo_root != repo_root.parent:
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "scripts"))

from ml_utils import MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES, normalize, percentile_rank

feature_path = repo_root / "data" / "processed" / "refresh_feature_vector.csv"
frame = pd.read_csv(feature_path)
frame = frame.dropna(subset=["is_declining_label"])

feature_cols = [col for col in MODEL_NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES if col in frame.columns]

# original row-random split for comparison
train_orig = frame.sample(frac=0.8, random_state=42)
test_orig = frame.drop(train_orig.index)

# client-aware split
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(frame, groups=frame["client_id"]))
train_group = frame.iloc[train_idx].reset_index(drop=True)
test_group = frame.iloc[test_idx].reset_index(drop=True)

# feature encoder helper

def build_matrix(df):
    numeric = df[[c for c in MODEL_NUMERIC_FEATURES if c in df.columns]].apply(pd.to_numeric, errors="coerce").replace([pd.NA], 0).fillna(0)
    categorical = df[[c for c in MODEL_CATEGORICAL_FEATURES if c in df.columns]].fillna("unknown").astype(str)
    cat_encoded = pd.get_dummies(categorical, prefix=MODEL_CATEGORICAL_FEATURES, dummy_na=False, dtype=float)
    matrix = pd.concat([numeric.reset_index(drop=True), cat_encoded.reset_index(drop=True)], axis=1)
    return matrix

X_orig_train = build_matrix(train_orig)
X_orig_test = build_matrix(test_orig)
X_orig_train, X_orig_test = X_orig_train.align(X_orig_test, join="outer", axis=1, fill_value=0)

X_group_train = build_matrix(train_group)
X_group_test = build_matrix(test_group)
X_group_train, X_group_test = X_group_train.align(X_group_test, join="outer", axis=1, fill_value=0)

models = {
    "random_forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "decision_tree": DecisionTreeClassifier(max_depth=5, random_state=42),
    "logistic_regression": LogisticRegression(max_iter=2000, random_state=42),
}

results = []
for name, model in models.items():
    model.fit(X_orig_train, train_orig["is_declining_label"].astype(int))
    orig_preds = model.predict(X_orig_test)
    orig_probs = model.predict_proba(X_orig_test)[:, 1]
    results.append({
        "model": name,
        "split": "row-random",
        "precision": precision_score(test_orig["is_declining_label"].astype(int), orig_preds, zero_division=0),
        "recall": recall_score(test_orig["is_declining_label"].astype(int), orig_preds, zero_division=0),
        "roc_auc": roc_auc_score(test_orig["is_declining_label"].astype(int), orig_probs),
    })
    
    model.fit(X_group_train, train_group["is_declining_label"].astype(int))
    group_preds = model.predict(X_group_test)
    group_probs = model.predict_proba(X_group_test)[:, 1]
    results.append({
        "model": name,
        "split": "client-aware",
        "precision": precision_score(test_group["is_declining_label"].astype(int), group_preds, zero_division=0),
        "recall": recall_score(test_group["is_declining_label"].astype(int), group_preds, zero_division=0),
        "roc_auc": roc_auc_score(test_group["is_declining_label"].astype(int), group_probs),
    })

pd.DataFrame(results)

/home/otto/Documents/projects/flyrank-internship/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/otto/Documents/projects/flyrank-internship/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as s

,model,split,precision,recall,roc_auc
0,random_forest,row-random,0.702207,0.765864,0.771289
1,random_forest,client-aware,0.583183,0.614481,0.609241
2,decision_tree,row-random,0.664711,0.778993,0.723550
3,decision_tree,client-aware,0.593699,0.574468,0.594308
4,logistic_regression,row-random,0.653653,0.774617,0.711698
5,logistic_regression,client-aware,0.574895,0.737377,0.616044


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [4]:
# Goal: audit the final feature set for leakage from label-derived columns.
import pandas as pd
from pathlib import Path
import sys

repo_root = Path.cwd()
while repo_root.name != "flyrank-internship" and repo_root != repo_root.parent:
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "scripts"))
from ml_utils import MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES

feature_path = repo_root / "data" / "processed" / "refresh_feature_vector.csv"
frame = pd.read_csv(feature_path)

leakage_columns = ["trend_direction", "trend_pct", "is_declining_label"]
used_features = [col for col in MODEL_NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES if col in frame.columns]

leakage_hits = [col for col in leakage_columns if col in used_features]

missing_in_data = [col for col in leakage_columns if col not in frame.columns]

{
    "used_feature_count": len(used_features),
    "leakage_columns_present_in_data": leakage_columns,
    "leakage_columns_used_as_features": leakage_hits,
    "missing_columns": missing_in_data,
}

{'used_feature_count': 26,
 'leakage_columns_present_in_data': ['trend_direction',
  'trend_pct',
  'is_declining_label'],
 'leakage_columns_used_as_features': [],
 'missing_columns': []}

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [5]:
# Goal: rewrite the boldest sentence into safe research language.
claim = (
    "A model trained on observable page and traffic signals can better identify pages at risk of future impression decline than a fixed rule."
)
safe_claim = (
    "We observed that a model using observable page and traffic signals had higher precision and recall "
    "for downstream decline risk than the baseline score in this dataset. This is a measured, directional, "
    "decision-support finding, not a causal guarantee."
)
claim, safe_claim

('A model trained on observable page and traffic signals can better identify pages at risk of future impression decline than a fixed rule.',
 'We observed that a model using observable page and traffic signals had higher precision and recall for downstream decline risk than the baseline score in this dataset. This is a measured, directional, decision-support finding, not a causal guarantee.')

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.